# Full Imbalance Comparison Report

Notebook nay tong hop day du pipeline: baseline khong balance vs balanced co M2M.
No tap trung vao 3 nhom ket qua: distribution, visualization mau, va evaluation.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

BASE_DIR = Path('..') / 'results' / 'full_imbalance_comparison'
VIS_DIR = BASE_DIR / 'visualizations'
SUMMARY_PATH = BASE_DIR / 'summary.json'

print('Base dir:', BASE_DIR.resolve())
print('Exists:', BASE_DIR.exists())

In [ ]:
summary = json.loads(SUMMARY_PATH.read_text())
df_counts = pd.read_csv(BASE_DIR / 'class_distribution_comparison.csv')
df_eval = pd.read_csv(BASE_DIR / 'evaluation_comparison.csv')
df_per_class = pd.read_csv(BASE_DIR / 'per_class_accuracy_comparison.csv')

display(Markdown('## Summary Metrics'))
display(pd.DataFrame(summary).T)
display(Markdown('## Class Distribution Comparison'))
display(df_counts)
display(Markdown('## Evaluation Comparison'))
display(df_eval)
display(Markdown('## Per-class Accuracy Comparison'))
display(df_per_class)

## 1. So luong data moi class truoc/sau balance

Bieu do nay giup nhin nhanh class nao rat thieu mau va class nao duoc M2M bo sung.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
x = range(len(df_counts))
width = 0.25
ax.bar([i - width for i in x], df_counts['baseline_before'], width=width, label='Baseline before')
ax.bar(x, df_counts['balanced_before'], width=width, label='Balanced before')
ax.bar([i + width for i in x], df_counts['balanced_after'], width=width, label='Balanced after M2M')
ax.set_xticks(list(x))
ax.set_xticklabels(df_counts['class_name'], rotation=30, ha='right')
ax.set_ylabel('Sample Count')
ax.set_title('Class Distribution: Before vs After Balance')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Visualization mau truoc va sau balance

Anh ben trai la raw source image; anh ben phai la ket qua sau M2M synthesis.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
before_img = Image(filename=str(VIS_DIR / 'samples_before_balance.png'))
after_img = Image(filename=str(VIS_DIR / 'samples_after_balance_m2m.png'))
before_img = plt.imread(str(VIS_DIR / 'samples_before_balance.png'))
after_img = plt.imread(str(VIS_DIR / 'samples_after_balance_m2m.png'))
axes[0].imshow(before_img)
axes[0].axis('off')
axes[0].set_title('Samples Before Balance')
axes[1].imshow(after_img)
axes[1].axis('off')
axes[1].set_title('Samples After Balance (M2M)')
plt.tight_layout()
plt.show()

## 3. Ket qua evaluation truoc va sau balance

So sanh standard accuracy va balanced accuracy de thay duoc tac dong that cua imbalance handling.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
metrics = df_eval.set_index('phase')
ax.bar(['Baseline Acc', 'Balanced Acc'], [metrics.loc['baseline', 'accuracy'], metrics.loc['balanced', 'accuracy']], label='Accuracy')
ax.bar(['Baseline Bal Acc', 'Balanced Bal Acc'], [metrics.loc['baseline', 'balanced_accuracy'], metrics.loc['balanced', 'balanced_accuracy']], label='Balanced Accuracy')
ax.set_ylim(0, 1)
ax.set_title('Evaluation Comparison')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
x = range(len(df_per_class))
width = 0.38
ax.bar([i - width/2 for i in x], df_per_class['baseline_accuracy'], width=width, label='Baseline')
ax.bar([i + width/2 for i in x], df_per_class['balanced_accuracy'], width=width, label='Balanced')
ax.set_xticks(list(x))
ax.set_xticklabels(df_per_class['class_name'], rotation=30, ha='right')
ax.set_ylim(0, 1)
ax.set_ylabel('Per-class accuracy')
ax.set_title('Per-class Accuracy Before vs After Balance')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Kết luận nhanh cho báo cáo/slide

- Baseline cho biet model bi bias boi majority classes.
- Balanced run voi M2M giup theo doi su dich chuyen per-class accuracy.
- Balanced accuracy la metric can uu tien khi trinh bay ket qua tren du lieu imbalanced.